# Optuna Hyperparameter Tuning Lab

Ví dụ tuning LightGBM classifier với Optuna + cross-validation, kèm logging kết quả.

In [ ]:
import optuna
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import StratifiedKFold, cross_val_score
from lightgbm import LGBMClassifier

## 1. Load data

In [ ]:
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target
X.shape, y.shape

## 2. Định nghĩa objective

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 200, 1000),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 16, 128),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10, log=True),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10, log=True),
        'random_state': 42,
        'objective': 'binary',
        'n_jobs': -1
    }
    model = LGBMClassifier(**params)
    scores = cross_val_score(model, X, y, cv=cv, scoring='roc_auc')
    return scores.mean()

## 3. Chạy study

In [ ]:
study = optuna.create_study(direction='maximize', study_name='lgbm_breast_cancer')
study.optimize(objective, n_trials=30)
print('Best ROC-AUC:', study.best_value)
study.best_params

## 4. Visualize kết quả

In [ ]:
optuna.visualization.plot_optimization_history(study)

In [ ]:
optuna.visualization.plot_param_importances(study)

## 5. Train final model & log

In [ ]:
best_model = LGBMClassifier(**study.best_params)
best_model.fit(X, y)
# TODO: log to MLflow/W&B tùy pipeline

> 💡 Gợi ý: kết hợp Optuna với MLflow (`mlflow_callback`) để tracking auto. Dùng `study.trials_dataframe()` xuất bảng report.